# Notebook 05 — Coverage Phase Diagram

This notebook synthesizes Notebook 03 (NMF recovery) and Notebook 04 (SAE dilution).

**Robust behavior:** if the expected CSV summaries are missing from `data/`, this notebook regenerates the needed summaries locally, then continues.

Outputs are saved as SVG-only figures plus CSV summaries.

In [ ]:
# Setup
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

MOD = 30
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
N_LANES = len(VALID_LANES_MOD30)

def save_svg(fig, name):
    path = FIG_DIR / f"{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

print("Notebook 05 setup complete.")
print("data/ CSVs currently visible:", sorted([p.name for p in DATA_DIR.glob('*.csv')]))

## 1. Shared data + metric helpers

In [ ]:
def make_batch_matrix(n_batches=400, batch_size=250, constrained=True, seed=9423):
    rng = np.random.default_rng(seed)
    all_numbers = np.arange(2, 100_000)
    valid_numbers = all_numbers[np.isin(all_numbers % MOD, VALID_LANES_MOD30)]

    rows = []
    sample_space = valid_numbers if constrained else all_numbers

    for _ in range(n_batches):
        sample = rng.choice(sample_space, size=batch_size, replace=True)
        counts = np.bincount(sample % MOD, minlength=MOD)
        rows.append(counts)

    X = np.array(rows, dtype=np.float32)
    X = X / X.sum(axis=1, keepdims=True)
    return X

lane_mask = np.zeros(MOD, dtype=bool)
lane_mask[VALID_LANES_MOD30] = True

def lane_mass_ratio(component):
    component = np.maximum(np.asarray(component), 0)
    total = component.sum()
    if total <= 1e-12:
        return 0.0
    return float(component[lane_mask].sum() / total)

def coverage_from_components(components):
    peaks = []
    for comp in components:
        comp = np.maximum(np.asarray(comp), 0)
        if comp.sum() <= 1e-12:
            continue
        peak = int(np.argmax(comp))
        if peak in VALID_LANES_MOD30:
            peaks.append(peak)
    return len(set(peaks)) / N_LANES

def structure_quality(coverage, lane_ratio, redundant, dead):
    return coverage * lane_ratio * (1 / (1 + redundant)) * (1 / (1 + dead))

## 2. Load or regenerate NMF summary

In [ ]:
def load_or_make_nmf_summary():
    path = DATA_DIR / "nmf_recovery_summary.csv"
    if path.exists():
        print(f"Loading existing {path}")
        return pd.read_csv(path)

    print("Missing data/nmf_recovery_summary.csv — regenerating NMF summary locally.")
    from sklearn.decomposition import NMF
    from sklearn.metrics import mean_squared_error

    X = make_batch_matrix(n_batches=400, batch_size=250, constrained=True, seed=9423)
    records = []

    for k in range(1, 13):
        model = NMF(n_components=k, init="nndsvda", random_state=9423, max_iter=2000)
        W = model.fit_transform(X)
        H = model.components_
        X_hat = W @ H

        peaks = []
        for h in H:
            h_pos = np.maximum(h, 0)
            if h_pos.sum() > 1e-12:
                peaks.append(int(np.argmax(h_pos)))
        valid_peaks = [p for p in peaks if p in VALID_LANES_MOD30]
        unique_valid = sorted(set(valid_peaks))

        records.append({
            "k": k,
            "reconstruction_mse": mean_squared_error(X, X_hat),
            "mean_lane_mass_ratio": np.mean([lane_mass_ratio(h) for h in H]),
            "coverage": len(unique_valid) / N_LANES,
            "unique_valid_lanes": len(unique_valid),
            "redundant_valid_features": max(len(valid_peaks) - len(unique_valid), 0),
            "dead_features": 0,
        })

    nmf = pd.DataFrame(records)
    nmf.to_csv(path, index=False)
    print(f"Saved regenerated {path}")
    return nmf

nmf_raw = load_or_make_nmf_summary()
nmf_raw.head()

## 3. Load or regenerate SAE summary

In [ ]:
def load_or_make_sae_summary():
    path = DATA_DIR / "sae_dilution_summary.csv"
    if path.exists():
        print(f"Loading existing {path}")
        return pd.read_csv(path)

    print("Missing data/sae_dilution_summary.csv — regenerating SAE summary locally.")
    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
    except Exception as exc:
        raise ImportError("PyTorch is required to regenerate SAE summaries. Run Notebook 04 first or upload sae_dilution_summary.csv.") from exc

    class TopKSAE(nn.Module):
        def __init__(self, input_dim=30, hidden_dim=16, topk=2):
            super().__init__()
            self.encoder = nn.Linear(input_dim, hidden_dim)
            self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
            self.topk = topk

        def topk_activation(self, z):
            z_relu = F.relu(z)
            values, indices = torch.topk(z_relu, self.topk, dim=1)
            mask = torch.zeros_like(z_relu)
            mask.scatter_(1, indices, 1.0)
            return z_relu * mask

        def forward(self, x):
            z = self.encoder(x)
            z_sparse = self.topk_activation(z)
            x_hat = F.relu(self.decoder(z_sparse))
            return x_hat, z_sparse

    def train_sae(X, hidden_dim=16, topk=2, epochs=500, lr=1e-2, seed=9423):
        torch.manual_seed(seed)
        model = TopKSAE(input_dim=X.shape[1], hidden_dim=hidden_dim, topk=topk)
        x = torch.tensor(X, dtype=torch.float32)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        losses = []

        for _ in range(epochs):
            opt.zero_grad()
            x_hat, z = model(x)
            loss = F.mse_loss(x_hat, x)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        with torch.no_grad():
            x_hat, z = model(x)

        decoder = model.decoder.weight.detach().cpu().numpy().T
        activations = z.detach().cpu().numpy()
        return decoder, activations, losses[-1]

    def summarize_sae(decoder, activations):
        active_counts = (activations > 1e-6).sum(axis=0)
        valid_peaks = []
        ratios = []

        for j, feature in enumerate(decoder):
            f = np.maximum(feature, 0)
            ratios.append(lane_mass_ratio(f))
            if f.sum() > 1e-12:
                peak = int(np.argmax(f))
                if peak in VALID_LANES_MOD30:
                    valid_peaks.append(peak)

        unique_valid = sorted(set(valid_peaks))
        return {
            "coverage": len(unique_valid) / N_LANES,
            "unique_valid_lanes": len(unique_valid),
            "redundant_valid_features": max(len(valid_peaks) - len(unique_valid), 0),
            "dead_features": int((active_counts == 0).sum()),
            "mean_lane_mass_ratio": float(np.mean(ratios)),
        }

    X = make_batch_matrix(n_batches=500, batch_size=250, constrained=True, seed=9423)
    records = []

    for hidden_dim in [8, 12, 16, 24, 32]:
        for topk in [1, 2, 4]:
            decoder, activations, final_loss = train_sae(
                X, hidden_dim=hidden_dim, topk=topk, epochs=500, lr=1e-2, seed=9423 + hidden_dim + topk
            )
            summary = summarize_sae(decoder, activations)
            summary.update({
                "hidden_dim": hidden_dim,
                "topk": topk,
                "final_loss": final_loss,
            })
            records.append(summary)

    sae = pd.DataFrame(records)
    sae.to_csv(path, index=False)
    print(f"Saved regenerated {path}")
    return sae

sae_raw = load_or_make_sae_summary()
sae_raw.head()

## 4. Normalize method tables

In [ ]:
# Normalize NMF columns
nmf = nmf_raw.copy()
if "k" in nmf.columns:
    nmf = nmf.rename(columns={"k": "capacity"})
if "mean_lane_mass_ratio" in nmf.columns:
    nmf = nmf.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})

nmf["method"] = "NMF"
nmf["topk"] = np.nan

if "coverage" not in nmf.columns:
    nmf["coverage"] = 1.0
if "dead_features" not in nmf.columns:
    nmf["dead_features"] = 0
if "redundant_valid_features" not in nmf.columns:
    nmf["redundant_valid_features"] = np.maximum(nmf["capacity"] - N_LANES, 0)

# Normalize SAE columns
sae = sae_raw.copy()
if "hidden_dim" in sae.columns:
    sae = sae.rename(columns={"hidden_dim": "capacity"})
if "mean_lane_mass_ratio" in sae.columns:
    sae = sae.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})
if "final_loss" in sae.columns and "reconstruction_mse" not in sae.columns:
    sae = sae.rename(columns={"final_loss": "reconstruction_mse"})

sae["method"] = "SAE"

cols = [
    "method",
    "capacity",
    "topk",
    "coverage",
    "lane_mass_ratio",
    "reconstruction_mse",
    "dead_features",
    "redundant_valid_features",
]

for col in cols:
    if col not in nmf.columns:
        nmf[col] = np.nan
    if col not in sae.columns:
        sae[col] = np.nan

df = pd.concat([nmf[cols], sae[cols]], ignore_index=True)

df["coverage"] = df["coverage"].fillna(0)
df["lane_mass_ratio"] = df["lane_mass_ratio"].fillna(0)
df["dead_features"] = df["dead_features"].fillna(0)
df["redundant_valid_features"] = df["redundant_valid_features"].fillna(0)

df["structure_quality"] = df.apply(
    lambda r: structure_quality(
        r["coverage"],
        r["lane_mass_ratio"],
        r["redundant_valid_features"],
        r["dead_features"],
    ),
    axis=1,
)

df.to_csv(DATA_DIR / "coverage_phase_diagram.csv", index=False)
df.head()

## 5. Coverage phase diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# NMF
nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(nmf_plot["capacity"], nmf_plot["coverage"], marker="o", label="NMF")

# SAE by top-k
for topk, group in df[df["method"] == "SAE"].groupby("topk"):
    g = group.sort_values("capacity")
    ax.plot(g["capacity"], g["coverage"], marker="o", label=f"SAE top-k={int(topk)}")

ax.set_title("Coverage Phase Diagram")
ax.set_xlabel("Capacity / dictionary size")
ax.set_ylabel("Valid lane coverage")
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.25)

save_svg(fig, "coverage_phase_diagram")
plt.show()

## 6. Alignment phase diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(nmf_plot["capacity"], nmf_plot["lane_mass_ratio"], marker="o", label="NMF")

for topk, group in df[df["method"] == "SAE"].groupby("topk"):
    g = group.sort_values("capacity")
    ax.plot(g["capacity"], g["lane_mass_ratio"], marker="o", label=f"SAE top-k={int(topk)}")

ax.set_title("Alignment Phase Diagram")
ax.set_xlabel("Capacity / dictionary size")
ax.set_ylabel("Mean lane-mass ratio")
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(True, alpha=0.25)

save_svg(fig, "alignment_phase_diagram")
plt.show()

## 7. Cost vs structure quality

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df.groupby("method"):
    ax.scatter(
        group["reconstruction_mse"],
        group["structure_quality"],
        s=30 + 4 * group["capacity"],
        alpha=0.75,
        label=method,
    )

ax.set_title("Cost vs Structure Quality")
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Structure quality")
ax.grid(True, alpha=0.25)
ax.legend()

save_svg(fig, "cost_vs_structure_quality")
plt.show()

## 8. Regime map

In [ ]:
def classify_regime(row):
    if row["coverage"] >= 0.99 and row["lane_mass_ratio"] >= 0.99 and row["dead_features"] == 0:
        return "recovered"
    if row["coverage"] < 0.75:
        return "fragmented"
    if row["dead_features"] > 0 or row["redundant_valid_features"] > 0:
        return "diluted"
    return "partial"

df["regime"] = df.apply(classify_regime, axis=1)
df.to_csv(DATA_DIR / "coverage_phase_diagram.csv", index=False)

regime_order = ["fragmented", "partial", "diluted", "recovered"]
regime_to_y = {r: i for i, r in enumerate(regime_order)}

fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df.groupby("method"):
    ax.scatter(
        group["capacity"],
        group["regime"].map(regime_to_y),
        s=40 + 120 * group["structure_quality"],
        alpha=0.75,
        label=method,
    )

ax.set_title("Method Regime Map")
ax.set_xlabel("Capacity / dictionary size")
ax.set_yticks(range(len(regime_order)))
ax.set_yticklabels(regime_order)
ax.grid(True, alpha=0.25)
ax.legend()

save_svg(fig, "method_regime_map")
plt.show()

## 9. Method comparison summary

In [ ]:
method_summary = (
    df.groupby("method")
    .agg(
        best_structure_quality=("structure_quality", "max"),
        best_coverage=("coverage", "max"),
        best_lane_mass_ratio=("lane_mass_ratio", "max"),
        min_reconstruction_mse=("reconstruction_mse", "min"),
        mean_dead_features=("dead_features", "mean"),
        mean_redundant_valid_features=("redundant_valid_features", "mean"),
    )
    .reset_index()
)

method_summary.to_csv(DATA_DIR / "method_comparison_summary.csv", index=False)
method_summary

## 10. Optional download cell

In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "05_coverage_phase_diagram_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)